# DreamerV3 on Kaggle: JAX/CUDA + Atari Pong

This notebook is structured for Kaggle runtime validation. It keeps GPU disabled in metadata for now, but the dependency path and gated smoke tests are prepared for JAX CUDA and DreamerV3 Atari/Pong.

In [ ]:
# Runtime switches for the Kaggle GPU validation runs.
RUN_CPU_DEBUG_SMOKE = False
RUN_GPU_ATARI_SMOKE = False
RUN_CRAFTER_IMPORT_CHECK = True
RUN_GPU_CRAFTER_SMOKE = True
SHOW_TENSORBOARD = False

# Keep source/dependencies outside /kaggle/working so Kaggle outputs stay small.
REPO_DIR = '/tmp/dreamerv3'
LOGDIR = '/kaggle/working/logdir'

# Native-reward Crafter run with lightweight action tracing.
SMOKE_STEPS = 1000
CRAFTER_SMOKE_STEPS = 100000
RESULT_RUN_NAME = 'crafter_native_100k_r64'
ACTION_TRACE_EVERY = 20


In [ ]:
import os
import platform
import shutil
import subprocess
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Kaggle kernel run type:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
print('Kaggle URL base:', os.environ.get('KAGGLE_URL_BASE'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Working directory:', os.getcwd())

if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi not found; this is expected before enabling a Kaggle GPU accelerator.')

In [ ]:
# Clone or update DreamerV3 outside /kaggle/working.
import os
import shutil
import subprocess

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/danijar/dreamerv3.git', REPO_DIR], check=True)
print('Repo ready at', REPO_DIR)

In [ ]:
%cd /tmp/dreamerv3
!python -m pip install -U pip setuptools wheel
!python -m pip install -U -r requirements.txt

In [ ]:
# Make Atari ROMs available for ALE/Gymnasium.
# The DreamerV3 requirements include autorom[accept-rom-license], but the ROM install step is explicit here
# so Kaggle logs show whether ROM setup succeeded.
!AutoROM --accept-license

In [ ]:
# Run dependency imports in a fresh Python process.
# Kaggle may preload numpy in the notebook kernel; after pip downgrades numpy, importing binary packages
# in this already-running process can produce ABI errors. A subprocess sees the installed versions cleanly.
import subprocess
import sys

code = r'''
import importlib
import importlib.metadata as metadata
import sys

modules = ['jax', 'numpy', 'ale_py', 'gymnasium', 'embodied', 'dreamerv3']
for name in modules:
    mod = importlib.import_module(name)
    version = getattr(mod, '__version__', 'unknown')
    print(f'{name}: {version}')

for dist in ['AutoROM', 'autorom.accept-rom-license']:
    try:
        print(f'{dist}:', metadata.version(dist))
    except metadata.PackageNotFoundError:
        print(f'{dist}: not installed')

print('subprocess executable:', sys.executable)
'''
subprocess.run([sys.executable, '-c', code], check=True)

In [ ]:
import subprocess
import sys

code = r'''
import jax

print('JAX version:', jax.__version__)
print('JAX default backend:', jax.default_backend())
print('JAX devices:', jax.devices())
try:
    print('JAX CUDA devices:', jax.devices('cuda'))
except Exception as exc:
    print('No CUDA backend available yet:', type(exc).__name__, exc)
'''
subprocess.run([sys.executable, '-c', code], check=True)

In [ ]:
# Atari/Pong smoke test in a fresh Python process.
# DreamerV3 uses its own ALE wrapper, so that is the required check. Gymnasium registration is only informational.
import subprocess
import sys

code = r'''
import numpy as np
import ale_py.roms as roms
from embodied.envs.atari import Atari

print('ALE pong ROM:', roms.get_rom_path('pong'))
env = Atari('pong', repeat=4, size=(96, 96), gray=True, noops=0, sticky=True, actions='all', seed=0)
obs = env.step({'reset': True, 'action': np.int32(0)})
print('DreamerV3 Atari wrapper OK')
print('  image shape:', obs['image'].shape, obs['image'].dtype)
print('  action space:', env.act_space['action'])

try:
    import gymnasium as gym
    import ale_py
    if hasattr(gym, 'register_envs'):
        gym.register_envs(ale_py)
    env2 = gym.make('ALE/Pong-v5', render_mode='rgb_array')
    obs2, info = env2.reset(seed=0)
    env2.close()
    print('Gymnasium ALE/Pong-v5 OK:', getattr(obs2, 'shape', None))
except Exception as exc:
    print('Gymnasium ALE/Pong-v5 optional check failed:', type(exc).__name__, exc)
'''
subprocess.run([sys.executable, '-c', code], check=True)

In [ ]:
# Optional: verify the Crafter dependency path without changing the current Atari/Pong target.
# This is useful because the final project direction is DreamerV3 on Crafter.
if RUN_CRAFTER_IMPORT_CHECK:
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'crafter'], check=True)
    code = r'''
import crafter
env = crafter.Env()
obs = env.reset()
print('crafter import/env OK')
print('crafter observation shape:', getattr(obs, 'shape', None))
'''
    subprocess.run([sys.executable, '-c', code], check=True)
else:
    print('Crafter import check skipped.')

In [ ]:
# CPU-only DreamerV3 debug smoke test. This intentionally uses the debug config, which sets jax.platform=cpu.
# Enable this only when you want to validate the training script path without consuming GPU quota.
if RUN_CPU_DEBUG_SMOKE:
    import subprocess
    import sys
    subprocess.run([
        sys.executable, 'dreamerv3/main.py',
        '--logdir', f'{LOGDIR}/atari_pong_cpu_debug',
        '--configs', 'atari', 'debug',
        '--task', 'atari_pong',
        '--run.steps', str(SMOKE_STEPS),
        '--run.envs', '1',
    ], check=True)
else:
    print('CPU debug smoke test skipped.')

In [ ]:
# GPU/JAX CUDA Atari/Pong smoke test.
# Before enabling this cell, set Kaggle accelerator to GPU and set RUN_GPU_ATARI_SMOKE=True above.
# Do not add the DreamerV3 debug config here because it forces jax.platform=cpu.
if RUN_GPU_ATARI_SMOKE:
    import subprocess
    import sys
    subprocess.run([
        sys.executable, 'dreamerv3/main.py',
        '--logdir', f'{LOGDIR}/atari_pong_gpu_1k',
        '--configs', 'atari',
        '--task', 'atari_pong',
        '--run.steps', str(SMOKE_STEPS),
        '--run.envs', '1',
        '--batch_size', '1',
        '--batch_length', '8',
        '--report_length', '8',
        '--replay_context', '0',
        '--jax.platform', 'cuda',
        '--jax.compute_dtype', 'float32',
        '--jax.prealloc', 'False',
    ], check=True)
else:
    print('GPU Atari/Pong smoke test skipped. Enable Kaggle GPU before running it.')

In [ ]:
# Patch the Crafter wrapper to write a lightweight action trace.
# It logs only every ACTION_TRACE_EVERY environment steps and never stores images.
if RUN_GPU_CRAFTER_SMOKE:
    from pathlib import Path

    crafter_path = Path(REPO_DIR) / 'embodied' / 'envs' / 'crafter.py'
    text = crafter_path.read_text(encoding='utf-8')

    if 'ACTION_TRACE_DIR' not in text:
        text = text.replace('import json\n', 'import json\nimport os\n')
        text = text.replace(
            '    self._done = True\n',
            """    self._done = True
    self._trace_every = int(os.environ.get('ACTION_TRACE_EVERY', '0') or 0)
    self._trace_path = None
    self._trace_file = None
    self._trace_seen_achievements = {}
    trace_dir = os.environ.get('ACTION_TRACE_DIR')
    if self._trace_every and trace_dir:
      trace_dir = elements.Path(trace_dir)
      trace_dir.mkdir()
      self._trace_path = trace_dir / 'action_trace.jsonl'
      self._trace_file = open(str(self._trace_path), 'a', buffering=1, encoding='utf-8')
      print(f'Writing Crafter action trace every {self._trace_every} steps: {self._trace_path}')
"""
        )
        text = text.replace(
            '      image = self._env.reset()\n      return self._obs(image, 0.0, {}, is_first=True)\n',
            """      image = self._env.reset()
      self._trace_seen_achievements = {}
      return self._obs(image, 0.0, {}, is_first=True)
"""
        )
        text = text.replace(
            '    self._reward += reward\n    self._length += 1\n',
            """    self._reward += reward
    self._length += 1
    self._write_action_trace(action['action'], reward, self._done, info)
"""
        )
        text = text.replace(
            '  def _write_stats(self, length, reward, info):\n',
            """  def _write_action_trace(self, action, reward, done, info):
    if not self._trace_file or not self._trace_every:
      return
    if self._length % self._trace_every:
      return
    achievements = info.get('achievements', {}) if info else {}
    events = [
        key for key, value in achievements.items()
        if value and not self._trace_seen_achievements.get(key, 0)]
    action_id = int(action)
    action_name = ''
    action_names = getattr(crafter.constants, 'actions', None)
    try:
      if isinstance(action_names, dict):
        action_name = str(action_names.get(action_id, ''))
      elif action_names is not None:
        action_name = str(action_names[action_id])
    except Exception:
      action_name = ''
    row = {
        'ep': int(self._episode),
        't': int(self._length),
        'action': action_id,
        'reward': float(reward),
        'done': bool(done),
        'event': ','.join(events),
    }
    if action_name:
      row['action_name'] = action_name
    self._trace_file.write(json.dumps(row, sort_keys=True) + '\\n')
    self._trace_seen_achievements = achievements.copy()

  def _write_stats(self, length, reward, info):
"""
        )
        crafter_path.write_text(text, encoding='utf-8')
        print('Patched Crafter action trace:', crafter_path)
    else:
        print('Crafter action trace patch already present:', crafter_path)
else:
    print('Crafter action trace patch skipped.')


In [ ]:
# GPU/JAX CUDA Crafter native-reward 100k training run with trace20 action logging.
# This uses Crafter's native reward and keeps train_ratio at 64 for a manageable validation run.
if RUN_GPU_CRAFTER_SMOKE:
    import os
    import subprocess
    import sys

    run_logdir = f'{LOGDIR}/{RESULT_RUN_NAME}'
    env = os.environ.copy()
    env['ACTION_TRACE_DIR'] = f'{run_logdir}/action_trace'
    env['ACTION_TRACE_EVERY'] = str(ACTION_TRACE_EVERY)

    subprocess.run([
        sys.executable, 'dreamerv3/main.py',
        '--logdir', run_logdir,
        '--configs', 'crafter',
        '--run.steps', str(CRAFTER_SMOKE_STEPS),
        '--run.envs', '1',
        '--run.train_ratio', '64',
        '--batch_size', '1',
        '--batch_length', '8',
        '--report_length', '8',
        '--replay_context', '0',
        '--jax.platform', 'cuda',
        '--jax.compute_dtype', 'float32',
        '--jax.prealloc', 'False',
    ], check=True, env=env)
else:
    print('GPU Crafter native-reward 100k trace20 training run skipped.')


In [ ]:
import subprocess
import sys
from IPython.display import Image, display

script = r'''
import json
from pathlib import Path

import matplotlib.pyplot as plt

logdir = Path("/kaggle/working/logdir") / "__RUN_NAME__"
metrics_path = logdir / "metrics.jsonl"
scores_path = logdir / "scores.jsonl"
trace_path = logdir / "action_trace" / "action_trace.jsonl"
plot_path = logdir / "loss_score_curves.png"

print("Run logdir:", logdir)

def load_jsonl(path):
    if not path.exists():
        print("Missing file:", path)
        return []
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f"Loaded {len(rows)} rows from {path.name}")
    return rows

def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)

metrics = load_jsonl(metrics_path)
scores = load_jsonl(scores_path)
trace_rows = load_jsonl(trace_path)

if trace_rows:
    print("\nLatest action trace rows:")
    for row in trace_rows[-5:]:
        print(json.dumps(row, ensure_ascii=False, sort_keys=True))
else:
    print("\nNo action trace rows yet. The run may not have reached 20 env steps.")

if not metrics:
    print("No metrics found. Run the training cell first.")
    raise SystemExit(0)

latest = metrics[-1]
loss_keys = sorted(
    key for key, value in latest.items()
    if key.startswith("train/loss/") and is_number(value)
)

print("\nLatest training losses:")
if loss_keys:
    for key in loss_keys:
        print(f"  {key}: {latest[key]:.6g}")
else:
    print("  No train/loss/* keys found.")

extra_keys = [
    key for key in ["replay/replay_ratio", "fps/policy", "fps/train"]
    if key in latest and is_number(latest[key])
]

if extra_keys:
    print("\nLatest runtime metrics:")
    for key in extra_keys:
        print(f"  {key}: {latest[key]:.6g}")

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)

all_loss_keys = sorted({
    key
    for row in metrics
    for key, value in row.items()
    if key.startswith("train/loss/") and is_number(value)
})

if all_loss_keys:
    for key in all_loss_keys:
        xs = []
        ys = []
        for index, row in enumerate(metrics):
            value = row.get(key)
            if is_number(value):
                xs.append(index)
                ys.append(value)
        axes[0].plot(xs, ys, marker="o", label=key.replace("train/loss/", ""))

    axes[0].set_title("Training Loss Curves")
    axes[0].set_xlabel("Log row")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=2, fontsize=8)
else:
    axes[0].text(0.5, 0.5, "No loss metrics found", ha="center", va="center")
    axes[0].set_axis_off()

score_rows = scores or metrics
score_keys = sorted({
    key
    for row in score_rows
    for key, value in row.items()
    if "score" in key.lower() and is_number(value)
})

if score_keys:
    print("\nLatest scores:")
    latest_score_row = score_rows[-1]
    for key in score_keys:
        if key in latest_score_row and is_number(latest_score_row[key]):
            print(f"  {key}: {latest_score_row[key]:.6g}")

    for key in score_keys:
        xs = []
        ys = []
        for index, row in enumerate(score_rows):
            value = row.get(key)
            if is_number(value):
                xs.append(index)
                ys.append(value)
        axes[1].plot(xs, ys, marker="o", label=key)

    axes[1].set_title("Score Curves")
    axes[1].set_xlabel("Log row")
    axes[1].set_ylabel("Score")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=8)
else:
    axes[1].text(
        0.5,
        0.5,
        "No score metrics yet. The short run may be too short to finish an episode.",
        ha="center",
        va="center",
    )
    axes[1].set_axis_off()
    print("\nNo score metrics found yet. A short run may be too short to finish an episode.")

fig.savefig(plot_path, dpi=140)
print("\nSaved plot:", plot_path)
'''

script = script.replace("__RUN_NAME__", RESULT_RUN_NAME)
subprocess.run([sys.executable, "-c", script], check=True)

from pathlib import Path
plot_path = Path(f"/kaggle/working/logdir/{RESULT_RUN_NAME}/loss_score_curves.png")
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print("No plot generated yet:", plot_path)
